# 3 · Checking every rule in one report

*Notebook 1.3: Step 3 of the session's four. In notebook 1.2, the guard learned OFAC's 50% rule and blocked the order for sanctions. In this notebook, we'll give it every rule and check them all in one report.*

**Cell 3.0 · Connect.** Each notebook runs in its own kernel, so each one starts by connecting,
with the same setup as Cell 0.1 in notebook 1.1. The cell then shows where the order stands after the
earlier notebooks. If a notebook was skipped, Connect puts the reference answers in place, so this one
still works.

In [ ]:
# Cell 3.0 · Connect
import sys, os; sys.path.insert(0, os.path.join(os.path.expanduser("~"), "SageMaker", "workshop", "helpers"))
from session import connect
connect(globals())
show_scoreboard()

> **What this showed.** The cell shows *Setup complete*, followed by the order's scoreboard so far, with one line for each
> step the order has been through. After a kernel restart, run this cell again.

In Step 2, one rule caught one of the order's four problems. Every rule has the same shape. It's a
query that returns a row, with its evidence, when the rule is broken, and returns nothing when the
rule holds. In this step, we'll read one complete rule, look at the one fact that deliberately isn't
kept in the graph, and then switch on every rule and send the order again.

**Cell 3.1 · One rule, as a query.** The rule we'll read in full is forced labour. The Uyghur Forced
Labor Prevention Act (UFLPA) bars goods made even in part by a company on its Entity List from import
into the United States, at any tier of the supply chain. Here's what each part of the query does.

- `MATCH (s:Supplier ...), (dest:Region ...) WHERE dest.country = $uflpa_country` limits the rule
  to U.S. deliveries. For any other region, nothing after this line runs, and no row comes back.
  `$uflpa_country` and `$uflpa_list` aren't written into the query. They come from
  `queries/rule_settings.json`, which ships with the queries, so a change to the rule's scope is a
  reviewed config change rather than a query edit.
- `MATCH tiers = (s)-[:SOURCES_FROM*1..]->(maker:Company)-[:ON_LIST]->(:RestrictedList ...)` walks
  down the supply chain, through any number of tiers, to a maker on the list. `tiers` is the whole
  path.
- `ORDER BY length(tiers) LIMIT 1` keeps the nearest match when there are several.
- `RETURN 'uflpa' AS rule, 'violation' AS severity, ... AS message, ... AS evidence` returns one row
  in the shape `block_message` reads, with the names along the path as `evidence` and the number of
  supply-chain tiers as `hops`. The path ends at the list, one step past the maker, so `hops` is
  `length(tiers) - 1`.

The cell runs the query twice, once for the order (US-East), and once for the same supplier
delivering to EU-Central.

*Predict: how many rows will each run return?*

In [ ]:
# Cell 3.1 · One rule, as a query
UFLPA_RULE = """
MATCH (s:Supplier {name: $supplier_name}), (dest:Region {name: $region_name})
WHERE dest.country = $uflpa_country
MATCH tiers = (s)-[:SOURCES_FROM*1..]->(maker:Company)-[:ON_LIST]->(:RestrictedList {name: $uflpa_list})
WITH maker, tiers ORDER BY length(tiers) LIMIT 1
// tiers ends at the list, not the maker: its SOURCES_FROM hops are length(tiers) - 1
RETURN 'uflpa' AS rule, 'violation' AS severity,
       maker.name + ', ' + toString(length(tiers) - 1) + ' tiers down, is on the UFLPA Entity List' AS message,
       [n IN nodes(tiers) | n.name] AS evidence, length(tiers) - 1 AS hops
"""
show_rule(UFLPA_RULE, [ORDER, dict(ORDER, region="EU-Central")])

> **What you should see.** For US-East, there's one row, and its evidence is drawn as the path the query matched, from
> AnyCompany Components along three `SOURCES_FROM` edges to AnyCompany Polysilicon Co., in red, and its
> `ON_LIST` edge to the UFLPA Entity List. For EU-Central, there's no row, because the rule holds there.
>
> **Why it matters.** Every rule in the workshop follows this template, with a variable-length pattern, a condition, and a
> row of evidence. The evidence is what makes a block explainable, because the model, and the person
> reading its reply, can see exactly why the order was stopped.

**Cell 3.2 · Where the stock count lives.** The fourth problem is stock, because AnyCompany
Components can't supply 40 units. Not every fact belongs in the graph. A stock count changes all day,
and AnyCompany already has a system of record for it, so the counts live in an Amazon DynamoDB table
keyed by `(warehouse_id, product_id)`. The graph only records which warehouse carries which product.

The stock rule therefore has two halves, and the cell shows both.

1. `run_query_file("stock_locations.cypher", ...)` asks the graph for the supplier's warehouses that
   carry the product, with their IDs.
2. `stock_levels(keys)` makes one DynamoDB `BatchGetItem` call for exactly those keys. The read is
   strongly consistent, so the count is the live one.

In [ ]:
# Cell 3.2 · Where the stock count lives
locations = run_query_file("stock_locations.cypher",
                           supplier_name=ORDER["supplier"], product_name=ORDER["product"])
keys = [(l["warehouse_id"], l["product_id"]) for l in locations]
units = stock_levels(keys)    # DynamoDB BatchGetItem, by key

show_stock(locations, units, ORDER)

> **What you should see.** The drawing shows the graph's half, the path from AnyCompany Components along `HAS_WAREHOUSE` to its
> one warehouse, in Amsterdam, and on along `STOCKS` to Server Hardware. On the warehouse is DynamoDB's
> half, the count stored under the warehouse's key, 25 units against the 40 ordered.
>
> **Why it matters.** The graph points at the right keys, and the system of record supplies the answer. When a count
> changes, nothing in the graph needs to change, because the guard reads the live number at the moment
> the agent acts.

**Cell 3.3 · Every rule, in one call.** Now we switch on every rule, covering the basics, sanctions,
forced labour, certification and stock, and ask for the full report in exactly the way your hook does
through `run_report`. Each rule is switched on by the list of installed rules, which is how Steps 1
and 2 ran with fewer of them.

**What runs, and where.**

- `validation_report(...)`, in [`helpers/verify.py`](../helpers/verify.py), is the function your hook calls
  through `run_report`. It reads both stores.
  - **The graph half** is one query, [`your-guard/queries/violations.cypher`](../your-guard/queries/violations.cypher). It finds the
    supplier, product and region once. Each rule that needs a walk, such as the ownership chain or
    the supply-chain tiers, runs in its own `CALL { }`. Then one `CASE` per rule turns its result
    into a row, only when the rule is installed (`'uflpa' IN $rules`) and broken. The threshold and
    the UFLPA's scope come from [`your-guard/queries/rule_settings.json`](../your-guard/queries/rule_settings.json) as parameters.
  - **The stock half** runs [`your-guard/queries/stock_locations.cypher`](../your-guard/queries/stock_locations.cypher) for the
    warehouses that carry the product, then `stock_levels()` reads their counts from DynamoDB with
    one `BatchGetItem`, keyed by the graph's IDs. If the total is short, it adds an `inventory` row.
- `show_report(...)`, in [`helpers/session.py`](../helpers/session.py), only displays: the table, and the evidence
  drawn as paths.

*Predict: how many rows will the report return, and which of them are violations?*

In [ ]:
# Cell 3.3 · Every rule, in one call
install_rules(ALL_RULES)

report = validation_report(ORDER["supplier"], ORDER["product"], ORDER["region"], ORDER["quantity"])
show_report(report, order=ORDER, native="Neptune's own view of these paths is the Graph tab in Cell 3.3a, next.")

> **What you should see.** The report returns four violations, and the drawing under the table shows each one's evidence as a
> path from the supplier. Sanctions runs up three `OWNED_BY` edges through blocked owners to OFAC's list.
> UFLPA runs down three `SOURCES_FROM` edges to a listed maker. Inventory ends at the Amsterdam
> warehouse, with 25 units against 40, and certification shows the supplier's `HOLDS_CERT` edge to
> NIST-800-171, marked expired, which US-East requires.
>
> **Why it matters.** A single call returns every reason at once. The certification rule is an ordinary set difference
> that SQL would handle just as well, and it's included in the same query so that the guard makes one
> call and gets one report.

**Cell 3.3a · The same evidence, in Neptune's graph view.** The drawing in Cell 3.3 comes from the
report's rows, laid out by the workshop's display code. This cell asks Neptune for the same evidence
as paths, and graph-notebook draws them in its **Graph** tab, the same native view as in Cells 0.3
and 2.1. The query finds the order's three vertices once, and each branch of `CALL { }` returns one
finding's path.

- **Sanctions** keeps the nearest ownership path on which every vertex is already `blocked`. The
  list comprehension counts the blocked vertices on the path, and only the list itself, at the end,
  isn't one.
- **Forced labor** keeps the nearest supply-chain path to a listed maker, and only when the region is
  in the United States.
- **Certification** keeps a certificate that the region requires and that the supplier's
  `HOLDS_CERT` edge doesn't mark `active`.
- **Stock** returns the warehouses that carry the product. Their counts are in DynamoDB, so this view
  shows where the stock is, and not how much.

*Predict: should the paths here match the rows in the drawing above, vertex for vertex?*

In [ ]:
%%oc -qp params -g blocked -d name -de {"OWNED_BY":"share_pct","HOLDS_CERT":"cert_status"} -l 40 -rel 40
// Cell 3.3a · The same evidence, in Neptune's graph view
MATCH (s:Supplier {name: $supplier_name}), (prod:Product {name: $product_name}),
      (r:Region {name: $region_name})
CALL {
  // Sanctions: the nearest chain of blocked owners up to the list
  WITH s
  MATCH p = (s)-[:OWNED_BY*1..]->(:Company)-[:SANCTIONED_BY]->(:SanctionsList)
  WHERE size([n IN nodes(p) WHERE n.blocked = true]) = length(p)
  RETURN p ORDER BY length(p) LIMIT 1
  UNION
  // Forced labor: the nearest listed maker, for a U.S. delivery
  WITH s, r
  MATCH p = (s)-[:SOURCES_FROM*1..]->(:Company)-[:ON_LIST]->(:RestrictedList {name: $uflpa_list})
  WHERE r.country = $uflpa_country
  RETURN p ORDER BY length(p) LIMIT 1
  UNION
  // Certification: required by the region, and not active for the supplier
  WITH s, r
  MATCH p = (s)-[h:HOLDS_CERT]->(:Certification)<-[:REQUIRES_CERT]-(r)
  WHERE h.cert_status <> 'active'
  RETURN p
  UNION
  // Stock: the warehouses that carry the product (the counts are in DynamoDB)
  WITH s, prod
  MATCH p = (s)-[:HAS_WAREHOUSE]->(:Warehouse)-[:STOCKS]->(prod)
  RETURN p
}
RETURN p

> **What you should see.** The Graph tab shows the same four paths as the drawing above. The chain through Holdings BV, Group SA
> and Industries OAO reaches OFAC's list, three `SOURCES_FROM` hops reach the UFLPA list, the
> `HOLDS_CERT` edge to NIST-800-171 says `expired`, and the Amsterdam warehouse carries Server Hardware.
> `-g blocked` colours the vertices the 50% rule blocked orange, and everything else blue.
>
> **Why it matters.** The drawing and the Graph tab are two views of the same facts. The drawing adds what Neptune doesn't
> hold, such as the stock count from DynamoDB and the rule each path belongs to, while the Graph tab is
> Neptune's own answer, which you can click into. A sanctions path is honest evidence only when every
> owner on it is blocked, and that's the condition you write in Go deeper G.3.

**Cell 3.4 · Send the order.** This cell runs the same lines as Cells 1.3 and 2.6, now with every
rule installed.

**What runs, and where.**

- The same functions as in Cell 1.3, with every rule installed by Cell 3.3.

*Predict: compared with Step 2, what will be different in what the model reads, and in what it
says?*

In [ ]:
# Cell 3.4 · Send the order
agent = build_agent(guard=NeptuneGuardPlugin(rules=reload(guard_rules)))
reply = ask(agent, HEADLINE)

record("Step 3", reply)

> **What you should see.** The trace shows `[BLOCKED]`, with all four reasons in what the model reads, followed by
> `recommend_alternatives`. The scoreboard now reads `Step 3   BLOCKED: 4 of 4 problems caught`.
>
> **Why it matters.** The model, the request, the tools and the code are all the same as in Step 1. Only the guard's
> rules have changed, and those rules are queries over the data rather than code in the agent.

<div class="alert alert-block alert-warning"><b>Stuck?</b> If there are fewer than four reasons, run Cell 3.3 again, because it installs every rule. If a
reason says <code>ownership-rule</code>, run <code>catch_up("ownership")</code> in a new cell, and then run this cell
again.</div>

**Cell 3.5 · The list changes.** Sanctions lists change every week, and a rule computed ahead of time
is only correct until they do. This cell walks through one change.

1. `oc(DESIGNATION)` has OFAC list AnyCompany Partners AS, which adds one new `SANCTIONED_BY` edge.
   AnyCompany Partners owns 50% of AnyCompany Electronics Group, where a sanctioned firm already
   holds 20%.
2. Before anything is recomputed, the cell shows the report for AnyCompany Electronics, whose 50%
   results are now out of date.
3. `apply_ownership_rule()` runs your query again, seed and rounds, against the new list.
4. The cell shows the report again, and how many suppliers are newly blocked.

The `finally` block undoes everything, so the rest of the session sees the original list.

*Predict: at step 2, before the recompute, should the report pass AnyCompany Electronics?*

In [ ]:
# Cell 3.5 · The list changes
DESIGNATION = """
MATCH (c:Company {name: 'AnyCompany Partners AS'}), (l:SanctionsList {name: 'OFAC SDN'})
MERGE (c)-[e:SANCTIONED_BY]->(l)
SET e.listed_date = '2026-10-08', e.program = 'WORKSHOP-DEMO'"""
ELECTRONICS = ["AnyCompany Electronics", "Server Hardware", "EU-Central", 10]

before = blocked_suppliers()
try:
    oc(DESIGNATION)
    show_report(validation_report(*ELECTRONICS), "Listed, not yet recomputed", order=ELECTRONICS,
                native="The designation is undone at the end of this cell, so it has no lasting Neptune view.")
    apply_ownership_rule()
    show_report(validation_report(*ELECTRONICS), "Recomputed against the new list", order=ELECTRONICS,
                native="The designation is undone at the end of this cell, so it has no lasting Neptune view.")
    show_note(f"One new edge: {blocked_suppliers() - before:,} more suppliers blocked, none of them on any list")
finally:
    undo_designation()

> **What you should see.** The first report shows an `ownership-rule` violation, saying that *the sanctions list has changed
> since the rule was computed*, and its drawing shows the 20% exposure path as a warning. After the
> recompute, AnyCompany Electronics is blocked, and the drawing shows the new chain, with blocked owners
> holding 70% of AnyCompany Electronics Group (20% plus the newly listed firm's 50%). Hundreds of other
> suppliers that aren't on any list are blocked with it.
>
> **Why it matters.** A result computed ahead of time can go stale. The report checks for that, and when the result is
> stale, it refuses to vouch for any supplier, which means it fails closed. No code changed here either,
> because the rules are queries over the data. In production, the list update itself would trigger the
> recompute.

<div class="alert alert-block alert-success"><b>In your system.</b> Write each rule as a query that returns a row <b>with evidence</b> when the rule is broken, and run
them all in one report. Keep fast-changing counts in their system of record, keyed so that the graph
can point at them. When a derived result is stale, fail closed.</div>

<div class="alert alert-block alert-info"><b>In production.</b> The other rules follow the same pattern, each with its own source. DHS publishes the
<a href="https://www.dhs.gov/uflpa-entity-list" target="_blank">UFLPA Entity List</a> as a web page and in Federal Register
notices, and it isn't part of the Consolidated Screening List, so teams load it separately.
Supply-chain tiers come from supplier declarations and licensed supply-chain mapping data.
Certifications come from the suppliers' own attestations, such as the NIST SP 800-171 scores that
defense contractors post to the government's Supplier Performance Risk System, and each one carries
an expiry date, as AnyCompany Components' does. Load each source on a schedule, record when it was
loaded, and fail closed when an input is older than it should be, as the guard does for the
sanctions list in Cell 3.5. Keep each raw download in Amazon S3 as a dated snapshot,
so every decision can be traced to the version of the list it was judged against, and keep the
rules' settings, such as the 50% threshold and the UFLPA's U.S. scope, in a reviewed config file, as
<code>queries/rule_settings.json</code> does here.</div>

**Next: open `1.4 - Testing and deploying the guardrail.ipynb`**, in this folder, for Step 4 · Testing and deploying the guardrail.

Your work is saved in `your-guard/`, so the next notebook picks it up when it connects.